In [ ]:
import pandas as pd
import matplotlib.pyplot as plt 
import seaborn as sns
import numpy as np
from itertools import combinations
from collections import Counter
from sklearn.preprocessing import MultiLabelBinarizer

import json

from settings import x_token, s_token, host, postgre_bd, user, psw
import funcs_yum as fy

In [ ]:
posgdb = fy.PostgresDB(host, postgre_bd, user, psw)

# main

In [ ]:
main_anime_df = pd.read_sql_query(
    '''
    SELECT *
    FROM anime_main AS m
        LEFT JOIN ratings AS r ON r.anime_id = m.anime_id
        LEFT JOIN ratings_distributions AS rd ON rd.anime_id = m.anime_id
    ORDER BY m.anime_id
    ''',
    posgdb.conn
)

main_copy = main_anime_df.copy()

In [ ]:
main_anime_df.info()

In [ ]:
main_anime_df.head()

## Всего на момент выгрузки на сайте было 10197 разных тайтлов 

## Распределение аниме по году выпуска
Тут можно указать две даты, после которы начинается рост количества выпускаемых аниме: 
- с 1985 начали выпускать аниме сразу на касетах без показа на больших экранах
- в 2000 наклыдывается множество факторов: переход к цифровой рисовке, развитие жанров для взрослой аудитории, 
    переход от длиносерийного производства (100+ серий) к более короткому формату (12/24 серий)

In [ ]:
mask = main_anime_df['year']>1970
data = main_anime_df[mask].groupby('year')['anime_id'].count()

n_bins = len(data)
sns.histplot(main_anime_df[mask]['year'], bins=n_bins)

plt.title('')
plt.show()

## Распределение аниме по периодам года 

In [ ]:
# season (зима, весна, лето, осень)
season = {1:'зима', 2:'весна', 3:'лето', 4:'осень'}

mask = main_anime_df['season']>0
data = main_anime_df[mask].groupby('season')['anime_id'].count()
data = data.rename(index=season)

n_bins = len(data)
plt.bar(data.index, data)

plt.title('')
plt.ylabel('количество')
plt.show()

## Распределение аниме по продолжительности (в секундах)

На распределениии по продолжительности выделяются три пика  
Эти пики можно охарактеризовать, если принять что в среднем в аниме сериале серии длятся по 24-25 минут  
тогда 3 группы это:
1) 34.000 - 38.000 секунд = 570 - 640 минут = 24/25 серии в среднем (с учетом средней длительности 1 серии)
2) 17.000 - 19.000 секунд = 280 - 320 минут = 12/13 серии в среднем
3) < 8000 = 130 минут 

Т.е. Среди всех аниме больше всего 12 и 24 серийников, + группа аниме длительностью менее двух часов (фильмы, короткие сериалы < 6 серий)

In [ ]:
mask1 = main_anime_df['duration']>0
mask2 = main_anime_df['duration']<100000

sns.histplot(main_anime_df['duration'].where(mask1 & mask2))

plt.title('')
plt.xlabel('длительность, сек')
plt.ylabel('количество, штук')
plt.show()

При детальном рассмотрении группы, в которой продолжительность < 8000 секунд  
так же можно выделить 3 группы:  
1) 5.000 - 7.000 секунд = 80 - 120 минут - это полнометражные фильмы
2) 2600 -3400 секунд = 40 - 60 минут - либо 2 серии спешиала, либо 1 более длинный выпуск
3) 1500 - 1800 секунд = 25 - 30 минут - 1 серийные выпуски (спешиалы, ответвления)

In [ ]:
mask1 = main_anime_df['duration']>000
mask2 = main_anime_df['duration']<10000

sns.histplot(main_anime_df['duration'].where(mask1 & mask2), bins=40)

plt.title('')
plt.xlabel('длительность, сек')
plt.ylabel('количество, штук')
plt.show()

И есть под 100 штук аниме, общая длительность которых более 24 часов, зачастую там несколько сотен серий  
из наиболее известных это Наруто и Блич

In [ ]:
mask1 = main_anime_df['duration']>1e5

main_anime_df[mask1][['title', 'duration']].sort_values(by='duration', ascending=False).head(7)

## Распределение аниме по количеству эпизодов

Из данных о количестве эпизодов в аниме подтвердаются вывводы сделанные на основании общей продолжительности в секундах:  
Наиболее часто выпускаются сериалы с 11/12/13 эпизодами, 24/25 серийные сериалы и полнометражные фильмы или 1-серийные дополнения

In [ ]:
mask = (main_anime_df['episodes_count'] > 0) & (main_anime_df['episodes_count'] < main_anime_df['episodes_count'].quantile(0.96)) 

sns.histplot(main_anime_df[mask]['episodes_count'])

plt.title('распределение по количеству эпизодов')
plt.xlabel('количество эпизодов в аниме')
plt.ylabel('количество штук')
plt.show()

## Распределение аниме по количеству простмотров страцы аниме

показывает популярность тех или иных аниме  
всего на сайте в сумме под 900 миллионов посещений разных аниме  
больше 1 миллиона посещений имеют 135 тайтлов, это 1.3% от всех аниме, а их сумма посещений составляет 35% всех посещений  
50% всех аниме посещали менее 20000 раз  

наибольшее число посещений на сайте у 3го сезона магической битвы (там более 11 миллионов мосещений)  
так же много посещений страни у аниме с большим количеством серий (тех же Наруто и Блича)

In [ ]:
print('общее количество посещений:', main_anime_df['views'].sum(), 'раз')

In [ ]:
mask1 = main_anime_df['views']>1e6

print('аниме с 1 миллионом посещений и больше')
print('доля от общего числа просмотров:', f'{main_anime_df['views'].where(mask1).sum() / main_anime_df['views'].sum():.4f}')
print('доля от общего числа аниме:', f'{main_anime_df['views'].where(mask1).count() / main_anime_df['views'].count():.4f}')

In [ ]:
mask1 = main_anime_df['views'] < 2e4

print('аниме с 20.000 посещений и менее')
print('имеют долю от общего числа просмотров:', f'{main_anime_df['views'].where(mask1).count() / main_anime_df['views'].count():.2f}')

In [ ]:
mask1 = main_anime_df['views'] > 0
mask2 = main_anime_df['views'] < 1e5

sns.histplot(main_anime_df['views'].where(mask1 & mask2))


plt.title('')
plt.xlabel('количество посещений')
plt.ylabel('количество аниме')
plt.show()

In [ ]:
mask1 = main_anime_df['views'] > 1e6

main_anime_df[mask1][['title', 'views']].sort_values(by='views', ascending=False).head(7)

## Распределение аниме по оценкам

Не у всех произведений стоит оценка:  
из 10197 наименований оценка стоит только у 9672 аниме  
(на сайте оценок нет у анонсированных, но еще не вышедших аниме)

In [ ]:
mask1 = main_anime_df['y_rating_avg'] > 0
print('количество аниме с оценкой выше 0:', main_anime_df['y_rating_avg'].where(mask1).count())

print('средняя оценка всех аниме на сайте:', main_anime_df['y_rating_avg'].where(mask1).mean())

Сравнение распределений оценок сайта yummi и MyAnimeList 

In [ ]:
mask1 = main_anime_df['year'] < 2027
mask2 = main_anime_df['y_rating_avg'] == 0
main_anime_df[mask1 & mask2][['title', 'year', 'views']]


In [ ]:
mask1 = main_anime_df['y_rating_avg'] > 0
mask2 = main_anime_df['myanimelist_rating'] > 0

sns.histplot(main_anime_df['y_rating_avg'].where(mask1), label='yummy rats')
sns.histplot(main_anime_df['myanimelist_rating'].where(mask2), alpha=0.3, label='MAL rats')

plt.title('распределение оценок аниме')
plt.xlabel('средняя оценка')
plt.ylabel('количество аниме')
plt.legend()
plt.show()

In [ ]:
main_anime_df['y_rating_int'] = main_anime_df['y_rating_avg'].astype(int)
main_anime_df["y_rating_rnd"] = main_anime_df["y_rating_avg"].round().astype(int)


In [ ]:
conditions = [
    main_anime_df["y_rating_avg"] < 5,
    main_anime_df["y_rating_avg"].between(5, 6, inclusive="left"),
    main_anime_df["y_rating_avg"] .between(6, 8, inclusive="left"),
    main_anime_df["y_rating_avg"].between(8, 9, inclusive="left"),
    main_anime_df["y_rating_avg"] > 9,
]

choices = [1, 2, 3, 4, 5]

main_anime_df["my_rat"] = np.select(conditions, choices, default=np.nan)

In [ ]:
mask1 = main_anime_df['y_rating_int'] > 0

ratings_1 = main_anime_df['y_rating_int'].where(mask1).value_counts()
ratings_2 = main_anime_df['y_rating_rnd'].where(mask1).value_counts()

sns.barplot(ratings_1, label='floor ratings')
sns.barplot(ratings_2, alpha=0.4, label='round ratings')

plt.title('дискретное распределение оценок')
plt.xlabel('средняя оценка')
plt.ylabel('количество аниме')
plt.legend()
plt.show()

Ниже график показывает как изменяется средняя оценка аниме в зависимости года выпуска   
До начала 2010 идет устойчивый тренд на увеличение средней оценки  
Пик достигается в 2008 году с значением в 7.06  
Примерно до 2015 средняя оценка за аниме остается на тех же значениях, около 6.95-7.00  
и после 2015 наблюдается тренд на снижение средних оценок  
(2026 не включен)

In [ ]:
mask = (main_anime_df['y_rating_int'] > 0) & (main_anime_df['year'] > 1970) & (main_anime_df['year'] < 2026)

sns.lineplot(data=main_anime_df[mask], x='year', y='y_rating_rnd', label='средняя оценка за год')

plt.title('динамика оценок по годам')
plt.xlabel('год выхода аниме')
plt.ylabel('средняя оценка')
plt.legend()
plt.show()

In [ ]:
mask = (main_anime_df['y_rating_int'] > 0) & (main_anime_df['year'] > 1970) & (main_anime_df['year'] < 2026)

main_anime_df[mask].groupby(by=['year'])['y_rating_rnd'].mean()

можно попробовать найти зависимость между оценкой аниме и количеством посещений  
У большинства аниме относительно малое количество посещений и оценки могут быть из всего диапазона значений  
С ростом количества посещений средняя оценка аниме растет, а самые низкие оценки зачастую у наименнее посещаемых аниме  

In [ ]:
mask = (main_anime_df['y_rating_avg'] > 0) & (main_anime_df['views'] < main_anime_df['views'].quantile(0.94))

plt.scatter(main_anime_df[mask]['views'], main_anime_df[mask]['y_rating_avg'], alpha=0.5)

plt.title('Диаграмма рассеяния средней оценки и количества посещений страницы аниме')
plt.xlabel('количество посещений страницы аниме')
plt.ylabel('средняя оценка')
# plt.legend()
plt.show()

Аналогичную тенденцию можно заметить и на количестве оставленных комментариев:  
Чем больше оставленно комментариев под аниме (тем оно популярнее) и средняя оценка выше у таких аниме  
и наоборот низшая оценка стоит зачастую у аниме с малым количеством комментариев  

In [ ]:
mask_v = (main_anime_df['y_rating_avg'] > 0) & (main_anime_df['count'] < main_anime_df['views'].quantile(0.95)) 

plt.scatter(main_anime_df['count'][mask_v], main_anime_df['y_rating_avg'][mask_v])

plt.title('Диаграмма рассеяния средней оценки и количества комментариев')
plt.xlabel('количество оставленных комментариев под аниме')
plt.ylabel('средняя оценка')
plt.show()

По статусу на сайте аниме делятся на вышедшие (9525 штук), анонсированные (528 штук) и те что выходят сейчас (144)

In [ ]:
x = main_anime_df['status_ru'].value_counts().index
y = main_anime_df['status_ru'].value_counts().values

plt.bar(x, y)

plt.title('распределение по статусам аниме')
plt.xlabel('статус')
plt.ylabel('количество штук аниме')
# plt.legend()
plt.show()

По типу на сайте аниме делятся на Сериал, ONA(без показ на тв), OVA(некое дополнение),'Полнометражные фильмы, Спешлы, Малометражный сериал, Короткометражный фильмы, 

In [ ]:
main_anime_df['type_ru'].value_counts()

In [ ]:
x = main_anime_df['type_ru'].value_counts().index
y = main_anime_df['type_ru'].value_counts().values

plt.bar(x, y)

plt.title('распределение по статусам аниме')
plt.xlabel('статус')
plt.xticks(rotation=90)
plt.ylabel('количество штук аниме')
# plt.legend()
plt.show()

# genres

In [ ]:
genres_df = pd.read_sql_query(
    '''
    SELECT *
    FROM anime_main AS m
        LEFT JOIN ratings AS r ON r.anime_id = m.anime_id
        LEFT JOIN ratings_distributions AS rd ON rd.anime_id = m.anime_id
        LEFT JOIN genres AS g on g.anime_id = m.anime_id
    ORDER BY m.anime_id
    ''',
    posgdb.conn
)

genres_copy = genres_df.copy()

In [ ]:
g_anime_id = genres_copy['anime_id'].iloc[:, 0]
genres_copy = genres_copy.drop(columns='anime_id')
genres_copy['anime_id'] = g_anime_id

## Частота встречаемости жанров аниме

In [ ]:
genr_rat = genres_copy.groupby('title_ru')['anime_id'].count().sort_values(ascending=False)
genr_rat = genr_rat[genr_rat>200]

sns.barplot(x=genr_rat.index, y=genr_rat, color='g', alpha=0.5)
plt.title('Топ жанров по количеству аниме')
plt.xlabel('жанр')
plt.ylabel('количество штук аниме')
plt.xticks(rotation=90)
plt.show()

Лидеры и атсудеры жанров по среднему рейтигу

In [ ]:
top_n = 15
last_n = 10
genr_rat = genres_copy.groupby('title_ru')['y_rating_avg'].mean().sort_values(ascending=False)
top = genr_rat.iloc[:top_n]
last = genr_rat.iloc[-last_n:]

fig, ax = plt.subplots(figsize=(10,5), nrows=1, ncols=2)

sns.barplot(x=top.index, y=top, ax=ax[0], color='g', alpha=0.5)
ax[0].set_title('распределение по статусам аниме')
ax[0].set_xlabel('жанр')
ax[0].tick_params(axis='x', rotation=90)
ax[0].set_ylabel('средний рейтинг')

sns.barplot(x=last.index, y=last, ax=ax[1], color='r', alpha=0.5)
ax[1].set_title('распределение по статусам аниме')
ax[1].set_xlabel('жанр')
ax[1].tick_params(axis='x', rotation=90)
ax[1].set_ylabel('средний рейтинг')


plt.show()

Тот же топ и ласт по рейтингу среди аниме только с учетом количества встречаемости жанра  
(оставляем только жанры кототрые встретились более 300 раз)

In [ ]:
top_n = 10
last_n = 10
genr_rat = genres_copy.groupby('title_ru').agg(y_rating_avg=('y_rating_avg', 'mean'),
        count=('y_rating_avg', 'count'))
genr_rat = genr_rat[genr_rat['count'] > 300]
genr_rat = genr_rat.sort_values(by='y_rating_avg', ascending=False)

top = genr_rat.iloc[:top_n]
last = genr_rat.iloc[-last_n:]
genr_rat
fig, ax = plt.subplots(figsize=(10,5), nrows=1, ncols=2)

sns.barplot(x=top.index, y=top['y_rating_avg'], ax=ax[0], color='g', alpha=0.5)
ax[0].set_title('распределение по статусам аниме')
ax[0].set_xlabel('жанр')
ax[0].tick_params(axis='x', rotation=90)
ax[0].set_ylabel('средний рейтинг')

sns.barplot(x=last.index, y=last['y_rating_avg'], ax=ax[1], color='r', alpha=0.5)
ax[1].set_title('распределение по статусам аниме')
ax[1].set_xlabel('жанр')
ax[1].tick_params(axis='x', rotation=90)
ax[1].set_ylabel('средний рейтинг')


plt.show()

## Разброс рейтингов по жанрам

тут можно увидеть что у всех 13 наиболее популярных жанров разброс средних оценок достаточно велик  
При этом самым стабильным жанром оказался 'Школьная жизнь', 
для этого жанра ширина интервала 3х сигм (среднее ± $3\sigma$) составляет 5.43 [от 4.42 до 9.95]

In [ ]:
genr_rat = genres_copy.groupby('title_ru')['anime_id'].count().sort_values(ascending=False)
genr_top13 = genr_rat[genr_rat>1000].index
genr_top13

genres_topn = genres_copy[(genres_copy['title_ru'].isin(genr_top13)) & (genres_copy['y_rating_avg'] > 0)]

fig, ax = plt.subplots(figsize=(10,5))

sns.boxplot(data=genres_topn, x='title_ru', y='y_rating_avg')
plt.title('Разброс рейтингов по жанрам')
plt.xlabel('жанр')
plt.xticks(rotation=90)
plt.ylabel('средний рейтинг')
plt.show()

## Тепловая карта совместной встречаемости жанров

In [ ]:
TOP_N = 20

top_genres = (genres_copy.groupby("genre_id")["anime_id"]
      .nunique()
      .sort_values(ascending=False)
      .head(TOP_N)
      .index
)

genres_topn = genres_copy[genres_copy["genre_id"].isin(top_genres)]
genre_dict = genres_topn[["genre_id", "title_ru"]].drop_duplicates().set_index("genre_id")["title_ru"].to_dict()

genres = (genres_topn.groupby('anime_id')["genre_id"].apply(list))

mlb = MultiLabelBinarizer()

binary = pd.DataFrame(
    mlb.fit_transform(genres),
    columns=mlb.classes_,
    index=genres.index
)

co_occurrence = binary.T @ binary

co_occurrence.rename(
    index=genre_dict,
    columns=genre_dict,
    inplace=True
)

plt.figure(figsize=(14,12))

mask = np.triu(np.ones_like(co_occurrence, dtype=bool))

sns.heatmap(
    co_occurrence,
    mask=mask,
    annot=True,
    fmt="d",
    cmap="YlOrRd", 
    square=True
)

plt.title("Матрица совместной встречаемости жанров")
plt.show()

In [ ]:
genres_copy

In [ ]:
genres_rat = genres_copy.groupby(['title_ru']).agg({'title_ru': 'count', 'year': 'max'})
# genres_rat = genres_rat.groupby('year')['title_ru'].max()
# genres_rat = genres_rat[genres_rat.index > 1970]
# plt.scatter(x=genres_rat.iloc[:, 0], y=genres_rat.iloc[:, 1])
genres_rat

# videos

In [ ]:
videos_df = pd.read_sql_query(
    '''
    SELECT *
    FROM videos
    ORDER BY anime_id
    ''',
    posgdb.conn
)

videos_copy = videos_df.copy()

In [ ]:
videos_copy.groupby('anime_id').count()

In [ ]:
videos_copy['episode_number'].value_counts()

# comments

In [ ]:
comments_df = pd.read_sql_query(
    '''
    SELECT *
    FROM comments
    ORDER BY anime_id
    ''',
    posgdb.conn
)

comments_copy = comments_df.copy()

In [ ]:
comments_copy.head()

In [ ]:
# comments_copy.to_csv(r'comments_data.csv', index=None)